Script requirements: 
- Convenient, configurable builds, 
- Convenient runtime parameter configuration, 
- Convenient result visualization

Style guide: 
- Organize cells as parameters -> definitions -> actions
- Use uppercase names for parameters and constants
- Use absolute paths for easier debugging



In [ ]:
import platform
import os
from os.path import dirname, exists, sep
import shutil

In [ ]:
# Set parameters

# Environment settings
CLEAN_OLD_DATA = True

# Build settings
BUILD_PARAM = {
    "CPU_NODE": 0,
    "CXL_NODE": 1 if platform.node() == "X99" else 2,
}

# Runtime settings
RUN_PARAM = {}

In [ ]:
# Constants

# Directories
CUR_DIR = os.getcwd()  # Current working directory
OUTPUT_DIR = CUR_DIR + sep + ".run"  # Output directory
ROOT_DIR = dirname(dirname(dirname(CUR_DIR)))  # Project root directory
BUILD_DIR = ROOT_DIR + sep + "build"  # Build directory

# Script names
BUILD_SH = OUTPUT_DIR + sep + "build.sh"
RUN_SH = OUTPUT_DIR + sep + "run.sh"

# Executable path
BIN = BUILD_DIR + sep + "benchmark" + sep + "ycsb" + sep + "ycsb"

In [ ]:
# Clean the environment
def check_environment():
    if CLEAN_OLD_DATA and exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)  # Remove existing output data

    os.mkdir(OUTPUT_DIR)  # Create the output directory

In [ ]:
# Generate the build script
def generate_build_script():
    cmake_command = (
        f"cmake -S {ROOT_DIR} -B {BUILD_DIR} \\\n"
        f"-DCMAKE_BUILD_TYPE:STRING=Release -DCMAKE_EXPORT_COMPILE_COMMANDS:BOOL=TRUE -G Ninja \\\n"
    )
    for k, v in BUILD_PARAM.items():  # Append build parameters to the CMake command
        cmake_command += f"-D{k}={v} \\\n"
    cmake_command += "\n"

    build_command = f"cmake --build {BUILD_DIR} --config Release --target all -j"

    with open(BUILD_SH, "w") as f:
        f.write("#!/bin/bash\n")
        f.write("set -x\n\n")
        f.write(cmake_command + "\n")
        f.write(build_command + "\n")

In [ ]:
# Generate the run script
def generate_run_script():
    commands = []

    command = f"{BIN} \\\n"
    for k, v in RUN_PARAM.items():
        command += f"--{k}={v} \\\n"
    command += "\n"

    commands.append(command)

    with open(RUN_SH, "w") as f:
        f.write("#!/bin/bash\n")
        f.write("set -x\n\n")
        for command in commands:
            f.write(command + "\n")

In [ ]:
# Prepare scripts
check_environment()
generate_build_script()
generate_run_script()

In [ ]:
# Build
!bash {BUILD_SH}

In [ ]:
# Run
!bash {RUN_SH}

In [ ]:
# Organize result data

In [ ]:
# Plot